# Non-Linear Economy Estimation (ANN)

Please note that this notebook uses a venv which points to a base python version of **3.13**, some functionality may be limited if using an older version of python.

## All Imports

In [26]:
from typing import Dict, Union, Tuple, Any, Optional, Sequence
import random
from dataclasses import dataclass
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.nn.utils import parameters_to_vector, vector_to_parameters
from torch.utils.data import TensorDataset
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [27]:
%pip install pandas numpy torch --quiet

Note: you may need to restart the kernel to use updated packages.


In [28]:
%matplotlib widget

## Data From Source Package

In [29]:
%pip install -e .. --quiet # This is broken for some reason still, figuring it out

Note: you may need to restart the kernel to use updated packages.


In [30]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
SRC = REPO_ROOT / "src"

sys.path.insert(0, str(SRC))

import autonomous_fed as afed

In [31]:
solver = afed.EnvironmentSolver(fred_key="7ab121fb17773e187bb6508e83e411da", override_device="cpu")

# Data Check
print (solver.historical_data.head())
print (solver.historical_data.tail())

             pi         y     i
date                           
1987Q3  2.66122 -0.388640  6.84
1987Q4  2.91876  0.526730  6.92
1988Q1  3.06577  0.260731  6.66
1988Q2  3.35274  0.788853  7.16
1988Q3  3.80207  0.595530  7.98
             pi         y     i
date                           
2006Q2  3.35642  1.319357  4.91
2006Q3  3.13805  0.978513  5.25
2006Q4  2.66316  1.380249  5.25
2007Q1  2.91019  1.210046  5.26
2007Q2  2.72883  1.336951  5.25


## Model Architecture

### Single Hidden Layer NARX Model

For our economy transition equations we have: ${y_t=\hat{f}^y(y_{t-1},y_{t-2},\pi_t,\pi_{t-1},\pi_{t-2},i_t,i_{t-1},i_{t-2})+\epsilon_t^y}$ and ${\pi_t=\hat{f}^\pi(y_t,y_{t-1},y_{t-2},\pi_{t-1},\pi_{t-2},i_t,i_{t-1},i_{t-2})+\epsilon_t^\pi}$

In this scenario our predictor function, ${\hat{f}}$ is an ANN (Artificial Neural Network) but it can be swapped with other nonlinear functions such as a sigmoid function or wavelet network. For our ANN predictor we have ${\hat{f}^m=b_0^m+\sum_{j=1}^h v_j^mG(\omega_j^{m'}s_t^m+b_j^m), m\in\{y,\pi}\}$

The Components of the ANN are as follows:
- ${m}$: ${\{y,\pi}\}$
- ${s_t^m}$: Input state vectors at time t.
- ${w_j^m}$: Weight vector for the j-th hidden neuron.
- ${b_j^m}$: Bias term for the current neuron.
- ${G(\cdot)}$: Activation function (nonlinear transform).
- ${v_j^m}$: Weight from hidden neuron ${j}$ to the output layer
- ${b_0^m}$: Bias at the output layer
- ${h}$: Number of hidden neurons.

As seen above the Neural Network type is a NARX Model with a single hidden layer and the activation function is the hyperbolic tangent therefore we define ${G(\cdot)}$ as follows: ${G(x)=tanh(x)=\frac{e^x-e^{-x}}{e^x+e^{-x}}}$

As done in the reference paper the ANNs are intialized with Nguyen-Widrow Initialization. In addition to this we use the Levenberg-Marquardt Algorithm for our optimizer.

## Model Buildout

in the following cells we will replicate the Bundesbank's Neural Net as close as possible by rebuilding some of the MATLAB tools in python for use with PyTorch.

Below we set the tensor dtype to float64 for pytorch because this model is being run locally on an ARM64 Macbook Pro. We can do training through the GPU by using MPS (Metal Performance Shaders) for our torch device but MPS does not support the float64 dtype only float32. As an effort to better replicate the MATLAB behavior from the reference paper we set dtype to float64 and bound our training to CPU for the torch device.

In [32]:
DTYPE = torch.float64

torch.use_deterministic_algorithms(True)

# (Optional if you ever use CUDA)
# torch.backends.cuda.matmul.allow_tf32 = False
# torch.backends.cudnn.allow_tf32 = False

#### MinMaxScaler

In [33]:
import torch
import torch.nn as nn

class TorchMinMaxScaler(nn.Module):
    """
    Torch-native MinMax scaler with registered buffers (state_dict + .to()).

    Maps per-feature x in [x_min, x_max] -> [a, b]:
        y = a + (x - x_min) * (b - a) / (x_max - x_min)

    - Fit over one or more reduction dims (default: dim=0, i.e., rows=samples, cols=features).
    - Safe for constant features (range ~ 0): maps them to midpoint of [a,b].
    - Works with float64 CPU (your notebook setup) and is deterministic given deterministic inputs.
    """

    def __init__(self, feature_range: tuple[float, float] = (-1.0, 1.0), eps: float = 1e-12) -> None:
        super().__init__()
        a, b = map(float, feature_range)
        if not (b > a):
            raise ValueError("feature_range must satisfy (max > min)")
        self.a = a
        self.b = b
        self.eps = float(eps)

        # Buffers populated by fit()
        self.register_buffer("x_min", torch.empty(0))
        self.register_buffer("x_max", torch.empty(0))
        self.register_buffer("scale", torch.empty(0))   # (b-a)/(x_max-x_min) (with safe handling)
        self.register_buffer("offset", torch.empty(0))  # a - scale*x_min

    @property
    def fitted(self) -> bool:
        return self.scale.numel() != 0

    @torch.no_grad()
    def fit(self, x: torch.Tensor, *, dim: int | tuple[int, ...] = 0) -> "TorchMinMaxScaler":
        if x.numel() == 0:
            raise ValueError("Cannot fit on an empty tensor.")
        if not x.dtype.is_floating_point:
            x = x.float()

        if isinstance(dim, int):
            dim = (dim,)
        # normalize negative dims
        dim = tuple(d if d >= 0 else x.ndim + d for d in dim)

        x_min = x.amin(dim=dim, keepdim=True)
        x_max = x.amax(dim=dim, keepdim=True)

        rng = (x_max - x_min)
        safe_rng = rng.clamp_min(self.eps)

        scale = (self.b - self.a) / safe_rng
        offset = self.a - scale * x_min

        # Constant features -> map to midpoint exactly
        const = rng <= self.eps
        if const.any():
            mid = 0.5 * (self.a + self.b)
            scale = torch.where(const, torch.zeros_like(scale), scale)
            offset = torch.where(const, torch.full_like(offset, mid), offset)

        self.x_min = x_min.detach()
        self.x_max = x_max.detach()
        self.scale = scale.detach()
        self.offset = offset.detach()
        return self

    def transform(self, x: torch.Tensor) -> torch.Tensor:
        if not self.fitted:
            raise RuntimeError("Scaler is not fitted. Call .fit(x) first.")
        return x * self.scale + self.offset

    def inverse_transform(self, y: torch.Tensor) -> torch.Tensor:
        if not self.fitted:
            raise RuntimeError("Scaler is not fitted. Call .fit(x) first.")
        inv = torch.where(self.scale == 0, torch.zeros_like(self.scale), 1.0 / self.scale)
        x = (y - self.offset) * inv
        return torch.where(self.scale == 0, self.x_min.expand_as(x), x)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.transform(x)

#### Nguyen-Widrow Initialization

In our reference paper it is noted that the ANNs are initialized using the Nguyen-Widrow Method. This is done because we are using ${\tanh(x)}$ for our activation function which has saturation zones near -1 and 1. In order to properly replicate the paper and ensure neurons are placed in the high-information region of the activation function,we build a Nguyen-Widrow Initializer as follows:

$$
{w_i=\beta\times\frac{w_i^0}{||w_i^0||}, b_i~U[-\beta,\beta], \beta=0.7n_{out}^{1/n_{in}}}
$$


In [34]:
import torch
import torch.nn as nn

@torch.no_grad()
def nguyen_widrow_(
    layer: nn.Linear,
    *,
    generator: torch.Generator | None = None,
    eps: float = 1e-12,
) -> nn.Linear:
    """
    In-place Nguyen-Widrow initialization for `nn.Linear`.

    Intended for MLPs with tanh/sigmoid hidden activations.
    Classic form:
      W ~ U(-0.5, 0.5)
      W_i <- beta * W_i / ||W_i||   (row-wise)
      b ~ U(-beta, beta)
      beta = 0.7 * (fan_out)^(1/fan_in)
    """
    if not isinstance(layer, nn.Linear):
        raise TypeError(f"nguyen_widrow_ expects nn.Linear, got {type(layer)!r}")

    fan_out, fan_in = layer.weight.shape
    if fan_in <= 0 or fan_out <= 0:
        raise ValueError(f"Invalid weight shape: {(fan_out, fan_in)}")

    beta = 0.7 * (fan_out ** (1.0 / fan_in))

    # Weights: U(-0.5,0.5) -> normalize rows -> scale by beta
    layer.weight.uniform_(-0.5, 0.5, generator=generator)
    norms = torch.linalg.vector_norm(layer.weight, ord=2, dim=1, keepdim=True).clamp_min(eps)
    layer.weight.div_(norms).mul_(beta)

    # Bias: U(-beta, beta)
    if layer.bias is not None:
        layer.bias.uniform_(-beta, beta, generator=generator)

    return layer


@torch.no_grad()
def apply_nguyen_widrow_(
    module: nn.Module,
    *,
    generator: torch.Generator | None = None,
) -> nn.Module:
    """Apply Nguyen-Widrow in-place to all `nn.Linear` submodules."""
    for m in module.modules():
        if isinstance(m, nn.Linear):
            nguyen_widrow_(m, generator=generator)
    return module

#### Single Hidden Layer Artificial Neural Network (NARX)

Below we mimic the refernce paper's Single-hidden-layer Artificial Neural Network with n-hidden nodes, ${\tanh(x)}$ activation function, and Nguyen-Widrow initialization.

In [35]:
import torch
from torch import nn

class SingleHiddenLayerNARX(nn.Module):
    """
    Torch-native single-hidden-layer feed-forward NARX core:
        y_hat_scaled = fc2(tanh(fc1(x_scaled)))

    Notes:
    - This module expects *already-scaled* inputs if you use scaling externally.
    - Uses your Nguyen-Widrow initializer for the hidden layer.
    """

    def __init__(
        self,
        in_features: int,
        hidden_features: int,
        out_features: int = 1,
        *,
        nguyen_widrow: bool = True,
        generator: torch.Generator | None = None,
        dtype: torch.dtype | None = None,
        device: torch.device | str | None = None,
    ) -> None:
        super().__init__()
        if in_features <= 0 or hidden_features <= 0 or out_features <= 0:
            raise ValueError("in_features, hidden_features, out_features must be positive.")

        self.in_features = int(in_features)
        self.hidden_features = int(hidden_features)
        self.out_features = int(out_features)

        self.fc1 = nn.Linear(self.in_features, self.hidden_features, bias=True, device=device, dtype=dtype)
        self.act = nn.Tanh()
        self.fc2 = nn.Linear(self.hidden_features, self.out_features, bias=True, device=device, dtype=dtype)

        self.reset_parameters(nguyen_widrow=nguyen_widrow, generator=generator)

    @torch.no_grad()
    def reset_parameters(
        self,
        *,
        nguyen_widrow: bool = True,
        generator: torch.Generator | None = None,
    ) -> None:
        # uses your notebook function nguyen_widrow_(layer, generator=...)
        if nguyen_widrow:
            nguyen_widrow_(self.fc1, generator=generator)
        else:
            self.fc1.reset_parameters()

        # stable output init
        nn.init.xavier_uniform_(self.fc2.weight, gain=1.0)
        if self.fc2.bias is not None:
            nn.init.zeros_(self.fc2.bias)

    def forward(self, x_scaled: torch.Tensor) -> torch.Tensor:
        if x_scaled.shape[-1] != self.in_features:
            raise ValueError(f"Expected last dim {self.in_features}, got {x_scaled.shape[-1]}")
        h = self.act(self.fc1(x_scaled))
        return self.fc2(h)

class ScaledNARX(nn.Module):
    """
    End-to-end "torch-native" NARX: raw_state -> scale -> core -> inverse_scale.

    Uses your:
      - TorchMinMaxScaler (buffers; state_dict-safe; .to() safe)
      - nguyen_widrow_ initializer
    """

    def __init__(
        self,
        in_features: int,
        hidden_features: int,
        out_features: int = 1,
        *,
        x_feature_range: tuple[float, float] = (-1.0, 1.0),
        y_feature_range: tuple[float, float] = (-1.0, 1.0),
        nguyen_widrow: bool = True,
        generator: torch.Generator | None = None,
        dtype: torch.dtype | None = None,
        device: torch.device | str | None = None,
    ) -> None:
        super().__init__()

        # Your scaler implementation (must already exist in the notebook)
        self.x_scaler = TorchMinMaxScaler(feature_range=x_feature_range)
        self.y_scaler = TorchMinMaxScaler(feature_range=y_feature_range)

        self.core = SingleHiddenLayerNARX(
            in_features=in_features,
            hidden_features=hidden_features,
            out_features=out_features,
            nguyen_widrow=nguyen_widrow,
            generator=generator,
            dtype=dtype,
            device=device,
        )

    @property
    def fitted(self) -> bool:
        return bool(self.x_scaler.fitted and self.y_scaler.fitted)

    @torch.no_grad()
    def fit_scalers(
        self,
        x_raw: torch.Tensor,
        y_raw: torch.Tensor,
        *,
        x_dim: int | tuple[int, ...] = 0,
        y_dim: int | tuple[int, ...] = 0,
    ) -> "ScaledNARX":
        """
        Fit x/y scalers on training data only.
        For (N, F) use dim=0. For (N, T, F) use dim=(0, 1).
        """
        self.x_scaler.fit(x_raw, dim=x_dim)
        self.y_scaler.fit(y_raw, dim=y_dim)
        return self

    def forward_scaled(self, x_scaled: torch.Tensor) -> torch.Tensor:
        """Scaled-in -> scaled-out (this is what LM/least-squares typically wants)."""
        return self.core(x_scaled)

    def forward(self, x_raw: torch.Tensor) -> torch.Tensor:
        """Raw-in -> raw-out (convenient for inference/plots)."""
        if not self.x_scaler.fitted:
            raise RuntimeError("x_scaler is not fitted. Call model.fit_scalers(...) first.")
        if not self.y_scaler.fitted:
            raise RuntimeError("y_scaler is not fitted. Call model.fit_scalers(...) first.")

        x_scaled = self.x_scaler.transform(x_raw)
        y_scaled = self.core(x_scaled)
        y_raw = self.y_scaler.inverse_transform(y_scaled)
        return y_raw

#### Levenberg-Marquardt Optimizer

In the Bundesbank paper, in addition to the Nguyen Widrow Initializer, the networks are trained using the Levenberg-Marquardt Algorithm, thus we build an optimizer to mimic this behavior.

In [36]:
from __future__ import annotations
import torch
from torch import nn
from torch.nn.utils import parameters_to_vector, vector_to_parameters
from torch.func import functional_call, jacrev

class LevenbergMarquardt(torch.optim.Optimizer):
    """
    Full-batch Levenberg-Marquardt (damped Gauss-Newton) for least-squares.

    closure must be: closure(params_dict) -> 1D residual vector r (pred - target).reshape(-1)

    This uses torch.func.jacrev to build the full Jacobian J (M,P):
        g   = J^T r
        A   = J^T J + (weight_decay + mu) I
        dθ  = -A^{-1} g
    """

    def __init__(
        self,
        model: nn.Module,
        params,
        *,
        mu: float = 1e-3,
        mu_dec: float = 0.1,
        mu_inc: float = 10.0,
        mu_max: float = 1e10,
        weight_decay: float = 0.0,
        max_trials: int = 25,
        eps: float = 1e-12,
    ) -> None:
        if mu <= 0:
            raise ValueError("mu must be > 0")
        defaults = dict(
            mu=float(mu),
            mu_dec=float(mu_dec),
            mu_inc=float(mu_inc),
            mu_max=float(mu_max),
            weight_decay=float(weight_decay),
            max_trials=int(max_trials),
            eps=float(eps),
        )
        super().__init__(params, defaults)
        self.model = model

    def _trainable_named_params(self) -> list[tuple[str, nn.Parameter]]:
        return [(n, p) for n, p in self.model.named_parameters() if p.requires_grad]

    @torch.no_grad()
    def _set_theta_(self, theta: torch.Tensor) -> None:
        ps = [p for _, p in self._trainable_named_params()]
        vector_to_parameters(theta, ps)

    def step(self, closure):
        group = self.param_groups[0]
        mu = float(group["mu"])
        mu_dec = float(group["mu_dec"])
        mu_inc = float(group["mu_inc"])
        mu_max = float(group["mu_max"])
        wd = float(group["weight_decay"])
        max_trials = int(group["max_trials"])
        eps = float(group["eps"])

        named = self._trainable_named_params()
        if not named:
            raise ValueError("No trainable parameters found.")

        ps = [p for _, p in named]
        names = [n for n, _ in named]

        theta0 = parameters_to_vector(ps).detach()
        P = int(theta0.numel())

        # Build a params pytree for torch.func
        params0 = {n: p.detach().clone().requires_grad_(True) for n, p in named}

        # Residuals at current point
        r0 = closure(params0).reshape(-1)
        if r0.ndim != 1:
            raise ValueError("closure(params_dict) must return a 1D residual vector.")
        M = int(r0.numel())
        sse0 = float((r0 @ r0).detach().cpu().item())

        # Full Jacobian via jacrev (returns dict[name] -> (M, *param.shape))
        J_tree = jacrev(lambda prm: closure(prm).reshape(-1))(params0)
        J = torch.cat([J_tree[n].reshape(M, -1) for n in names], dim=1)  # (M,P)

        g = J.T @ r0  # (P,)
        JTJ = J.T @ J  # (P,P)

        I = torch.eye(P, device=JTJ.device, dtype=JTJ.dtype)
        if wd != 0.0:
            JTJ = JTJ + wd * I

        accepted = False
        trials = 0
        sse1 = sse0

        while trials < max_trials and mu <= mu_max:
            trials += 1

            A = JTJ + mu * I
            rhs = -g

            try:
                dtheta = torch.linalg.solve(A, rhs)
            except RuntimeError:
                dtheta = torch.linalg.lstsq(A, rhs.unsqueeze(1)).solution.squeeze(1)

            theta1 = theta0 + dtheta
            self._set_theta_(theta1)

            # Evaluate new SSE (no grad)
            with torch.no_grad():
                params1 = {n: p.detach() for n, p in self.model.named_parameters() if p.requires_grad}
                r1 = closure(params1).reshape(-1)
                sse1 = float((r1 @ r1).cpu().item())

            if sse1 < sse0:
                accepted = True
                mu = max(1e-30, mu * mu_dec)
                break

            # reject -> restore and increase damping
            self._set_theta_(theta0)
            mu = mu * mu_inc

        group["mu"] = mu
        return {
            "accepted": accepted,
            "trials": trials,
            "mu": mu,
            "M": M,
            "P": P,
            "old_sse": sse0,
            "new_sse": sse1,
            "mu_max_hit": bool(mu > mu_max),
            "g_norm": float(torch.linalg.vector_norm(g).detach().cpu().item()),
            "step_norm": float(torch.linalg.vector_norm(dtheta).detach().cpu().item()) if "dtheta" in locals() else 0.0,
            "cond_A": float(torch.linalg.cond(JTJ + mu * I).detach().cpu().item() + eps),
        }

### Training, Searching, and Optimization

In [37]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.func import functional_call


def build_narx_state(
    df: pd.DataFrame,
    *,
    target_col: str,
    lags: int = 2,
    col_map: dict[str, str] | None = None,
) -> tuple[torch.Tensor, torch.Tensor, list[str]]:
    """
    Creates X_raw, y_raw (torch float64) for the Bundesbank-style NARX:

    For y_t:
      X = [y_{t-1}, y_{t-2}, pi_t, pi_{t-1}, pi_{t-2}, i_t, i_{t-1}, i_{t-2}]
      y = y_t

    For pi_t:
      X = [y_t, y_{t-1}, y_{t-2}, pi_{t-1}, pi_{t-2}, i_t, i_{t-1}, i_{t-2}]
      y = pi_t
    """
    if col_map is None:
        col_map = {}

    def col(name: str) -> str:
        return col_map.get(name, name)

    # auto-fallback for common naming
    if col("i") not in df.columns and "ffr" in df.columns and "i" not in col_map:
        col_map = dict(col_map)
        col_map["i"] = "ffr"

    for req in (col("y"), col("pi"), col("i")):
        if req not in df.columns:
            raise KeyError(f"Missing required column {req!r}. Provide col_map={{'i': '...'}} if needed.")

    work = pd.DataFrame(
        {
            "y": df[col("y")].astype(float).to_numpy(),
            "pi": df[col("pi")].astype(float).to_numpy(),
            "i": df[col("i")].astype(float).to_numpy(),
        }
    )

    # lags
    for k in range(1, lags + 1):
        work[f"y_lag{k}"] = pd.Series(work["y"]).shift(k)
        work[f"pi_lag{k}"] = pd.Series(work["pi"]).shift(k)
        work[f"i_lag{k}"] = pd.Series(work["i"]).shift(k)

    if target_col not in ("y", "pi"):
        raise ValueError("target_col must be 'y' or 'pi'.")

    if target_col == "y":
        feature_cols = ["y_lag1", "y_lag2", "pi", "pi_lag1", "pi_lag2", "i", "i_lag1", "i_lag2"]
        y_series = pd.Series(work["y"])
    else:
        feature_cols = ["y", "y_lag1", "y_lag2", "pi_lag1", "pi_lag2", "i", "i_lag1", "i_lag2"]
        y_series = pd.Series(work["pi"])

    mat = pd.concat([work[feature_cols], y_series.rename(target_col)], axis=1).dropna(axis=0)

    X_raw = torch.as_tensor(mat[feature_cols].to_numpy(np.float64), dtype=DTYPE)
    y_raw = torch.as_tensor(mat[[target_col]].to_numpy(np.float64), dtype=DTYPE)
    return X_raw, y_raw, feature_cols


def make_lm_residual_closure_core(
    model_core: nn.Module,
    X_scaled: torch.Tensor,
    y_scaled: torch.Tensor,
):
    def closure(params_dict: dict[str, torch.Tensor]) -> torch.Tensor:
        pred = functional_call(model_core, params_dict, (X_scaled,))
        return (pred - y_scaled).reshape(-1)
    return closure


@torch.no_grad()
def mse_raw_from_scaled_pred(
    y_pred_scaled: torch.Tensor,
    y_true_raw: torch.Tensor,
    *,
    y_scaler: TorchMinMaxScaler,
) -> float:
    y_pred_raw = y_scaler.inverse_transform(y_pred_scaled)
    return float(torch.mean((y_pred_raw - y_true_raw) ** 2).cpu().item())


def train_lm_trial(
    X_tr_raw: torch.Tensor,
    y_tr_raw: torch.Tensor,
    X_val_raw: torch.Tensor,
    y_val_raw: torch.Tensor,
    *,
    h: int,
    seed: int,
    mu: float = 1e-3,
    max_iters: int = 50,
    max_trials: int = 25,
    weight_decay: float = 0.0,
    device: str | torch.device = "cpu",
) -> dict[str, object]:
    """
    One LM training run (one random init) for a given h.
    Returns dict with model, scalers, and raw validation MSE.
    """
    device = torch.device(device)

    # Fit scalers on TRAIN only
    x_scaler = TorchMinMaxScaler(feature_range=(-1.0, 1.0))
    y_scaler = TorchMinMaxScaler(feature_range=(-1.0, 1.0))
    x_scaler.fit(X_tr_raw, dim=0)
    y_scaler.fit(y_tr_raw, dim=0)

    X_tr_s = x_scaler.transform(X_tr_raw).to(device=device)
    y_tr_s = y_scaler.transform(y_tr_raw).to(device=device)
    X_val_s = x_scaler.transform(X_val_raw).to(device=device)

    g = torch.Generator(device="cpu").manual_seed(int(seed))

    model = SingleHiddenLayerNARX(
        in_features=int(X_tr_raw.shape[1]),
        hidden_features=int(h),
        out_features=int(y_tr_raw.shape[1]),
        nguyen_widrow=True,
        generator=g,
        dtype=DTYPE,
        device=device,
    )

    closure = make_lm_residual_closure_core(model, X_tr_s, y_tr_s)
    opt = LevenbergMarquardt(
        model,
        model.parameters(),
        mu=mu,
        max_trials=max_trials,
        weight_decay=weight_decay,
    )

    for _ in range(int(max_iters)):
        info = opt.step(closure)
        if (not info["accepted"]) and info["mu_max_hit"]:
            break

    with torch.no_grad():
        y_val_pred_s = model(X_val_s)
        val_mse_raw = mse_raw_from_scaled_pred(y_val_pred_s, y_val_raw.to(device=device), y_scaler=y_scaler)

    return {
        "model": model,
        "x_transformer": x_scaler,
        "y_scaler": y_scaler,
        "val_mse_raw": float(val_mse_raw),
    }


def search_hidden_units_LM(
    df: pd.DataFrame,
    *,
    target_col: str,
    train_ratio: float = 0.85,
    hidden_min: int = 1,
    hidden_max: int = 10,
    n_trials: int = 30,
    base_seed: int = 0,
    mu: float = 1e-3,
    max_iters: int = 50,
    device: str | torch.device = "cpu",
    col_map: dict[str, str] | None = None,
) -> dict[str, object]:
    """
    Your requested procedure:
      - split 85/15
      - h in [1..10]
      - 30 random inits per h
      - choose h* minimizing avg validation MSE (raw) across trials

    Returns:
      {
        "h": h_star,
        "metric": avg_val_mse_raw_for_h_star,
        "model": best_trial_model_within_h_star,   # expects x_scaled -> y_scaled
        "meta": {... scalers/state ...},
        "curve": {h: {"avg":..., "std":..., "all": [...]}, ...}
      }
    """
    X_raw, y_raw, feature_cols = build_narx_state(df, target_col=target_col, lags=2, col_map=col_map)

    N = int(X_raw.shape[0])
    n_tr = int(np.floor(N * float(train_ratio)))
    if n_tr <= 2 or n_tr >= N:
        raise ValueError(f"Bad split: N={N}, train_ratio={train_ratio} -> n_tr={n_tr}")

    X_tr_raw, y_tr_raw = X_raw[:n_tr], y_raw[:n_tr]
    X_val_raw, y_val_raw = X_raw[n_tr:], y_raw[n_tr:]

    curve: dict[int, dict[str, object]] = {}
    best_h = None
    best_h_avg = float("inf")
    best_trial_payload = None

    for h in range(int(hidden_min), int(hidden_max) + 1):
        trial_mses: list[float] = []
        best_trial_mse = float("inf")
        best_payload_h = None

        for t in range(int(n_trials)):
            # deterministic unique seed per (h, t)
            seed = int(base_seed) + 10_000 * int(h) + int(t)

            payload = train_lm_trial(
                X_tr_raw, y_tr_raw, X_val_raw, y_val_raw,
                h=h,
                seed=seed,
                mu=mu,
                max_iters=max_iters,
                device=device,
            )

            m = float(payload["val_mse_raw"])
            trial_mses.append(m)
            if m < best_trial_mse:
                best_trial_mse = m
                best_payload_h = payload

        avg = float(np.mean(trial_mses))
        std = float(np.std(trial_mses, ddof=1)) if len(trial_mses) > 1 else 0.0

        curve[h] = {"avg": avg, "std": std, "all": trial_mses, "best": best_trial_mse}

        if avg < best_h_avg:
            best_h_avg = avg
            best_h = h
            best_trial_payload = best_payload_h

    assert best_h is not None and best_trial_payload is not None

    return {
        "h": int(best_h),
        "metric": float(best_h_avg),  # avg val MSE (raw) across 30 trials at best_h
        "model": best_trial_payload["model"],  # best trial model within best_h
        "meta": {
            "state_full_raw": X_raw,
            "target_full_raw": y_raw,
            "state_feature_cols": feature_cols,
            "n_tr": n_tr,
            "x_transformer": best_trial_payload["x_transformer"],
            "y_scaler": best_trial_payload["y_scaler"],
        },
        "curve": curve,
    }


def search_both_targets_LM(
    df: pd.DataFrame,
    *,
    train_ratio: float = 0.85,
    hidden_min: int = 1,
    hidden_max: int = 10,
    n_trials: int = 30,
    base_seed: int = 0,
    mu: float = 1e-3,
    max_iters: int = 50,
    device: str | torch.device = "cpu",
    col_map: dict[str, str] | None = None,
) -> dict[str, dict[str, object]]:
    """
    Runs the above search for BOTH:
      - output gap (y)
      - inflation (pi)
    """
    out_y = search_hidden_units_LM(
        df,
        target_col="y",
        train_ratio=train_ratio,
        hidden_min=hidden_min,
        hidden_max=hidden_max,
        n_trials=n_trials,
        base_seed=base_seed,
        mu=mu,
        max_iters=max_iters,
        device=device,
        col_map=col_map,
    )
    out_pi = search_hidden_units_LM(
        df,
        target_col="pi",
        train_ratio=train_ratio,
        hidden_min=hidden_min,
        hidden_max=hidden_max,
        n_trials=n_trials,
        base_seed=base_seed + 999_999,  # de-correlate seeds across targets
        mu=mu,
        max_iters=max_iters,
        device=device,
        col_map=col_map,
    )
    return {"y": out_y, "pi": out_pi}

@torch.no_grad()
def _overall_mse_raw_for_payload(
    payload: dict[str, object],
    X_tr_raw: torch.Tensor,
    y_tr_raw: torch.Tensor,
    X_val_raw: torch.Tensor,
    y_val_raw: torch.Tensor,
    *,
    device: str | torch.device = "cpu",
) -> tuple[float, float, float]:
    """Returns (train_mse_raw, val_mse_raw, overall_mse_raw)."""
    model = payload["model"]
    x_scaler = payload["x_transformer"]
    y_scaler = payload["y_scaler"]

    device = torch.device(device)

    X_tr_s = x_scaler.transform(X_tr_raw).to(device=device)
    X_val_s = x_scaler.transform(X_val_raw).to(device=device)

    y_tr_pred_s = model(X_tr_s)
    y_val_pred_s = model(X_val_s)

    tr_mse = mse_raw_from_scaled_pred(y_tr_pred_s, y_tr_raw.to(device=device), y_scaler=y_scaler)
    va_mse = mse_raw_from_scaled_pred(y_val_pred_s, y_val_raw.to(device=device), y_scaler=y_scaler)

    # overall MSE computed on concatenated (train + val)
    y_all_raw = torch.cat([y_tr_raw, y_val_raw], dim=0).to(device=device)
    y_all_pred_s = torch.cat([y_tr_pred_s, y_val_pred_s], dim=0)
    ov_mse = mse_raw_from_scaled_pred(y_all_pred_s, y_all_raw, y_scaler=y_scaler)

    return float(tr_mse), float(va_mse), float(ov_mse)


def retrain_fixed_h_pick_best_overall(
    df: pd.DataFrame,
    *,
    target_col: str,
    h_fixed: int,
    train_ratio: float = 0.85,
    n_trials: int = 30,
    base_seed: int = 0,
    mu: float = 1e-3,
    max_iters: int = 50,
    device: str | torch.device = "cpu",
    col_map: dict[str, str] | None = None,
) -> dict[str, object]:
    """
    Fix h, retrain 30 times, pick the run with the lowest OVERALL MSE (raw)
    over train+val combined.
    """
    X_raw, y_raw, feature_cols = build_narx_state(df, target_col=target_col, lags=2, col_map=col_map)

    N = int(X_raw.shape[0])
    n_tr = int(np.floor(N * float(train_ratio)))
    if n_tr <= 2 or n_tr >= N:
        raise ValueError(f"Bad split: N={N}, train_ratio={train_ratio} -> n_tr={n_tr}")

    X_tr_raw, y_tr_raw = X_raw[:n_tr], y_raw[:n_tr]
    X_val_raw, y_val_raw = X_raw[n_tr:], y_raw[n_tr:]

    best_payload = None
    best_overall = float("inf")
    all_rows = []

    for t in range(int(n_trials)):
        seed = int(base_seed) + 10_000 * int(h_fixed) + int(t)

        payload = train_lm_trial(
            X_tr_raw, y_tr_raw, X_val_raw, y_val_raw,
            h=int(h_fixed),
            seed=seed,
            mu=mu,
            max_iters=max_iters,
            device=device,
        )

        tr_mse, va_mse, ov_mse = _overall_mse_raw_for_payload(
            payload, X_tr_raw, y_tr_raw, X_val_raw, y_val_raw, device=device
        )

        all_rows.append(
            {"seed": int(seed), "train_mse_raw": tr_mse, "val_mse_raw": va_mse, "overall_mse_raw": ov_mse}
        )

        if ov_mse < best_overall:
            best_overall = ov_mse
            best_payload = payload
            best_payload["_picked_seed"] = int(seed)
            best_payload["_picked_train_mse_raw"] = float(tr_mse)
            best_payload["_picked_val_mse_raw"] = float(va_mse)
            best_payload["_picked_overall_mse_raw"] = float(ov_mse)

    assert best_payload is not None

    return {
        "h": int(h_fixed),
        "model": best_payload["model"],  # trained model (x_scaled -> y_scaled)
        "meta": {
            "state_full_raw": X_raw,
            "target_full_raw": y_raw,
            "state_feature_cols": feature_cols,
            "n_tr": n_tr,
            "x_transformer": best_payload["x_transformer"],
            "y_scaler": best_payload["y_scaler"],
            "picked_seed": best_payload["_picked_seed"],
            "picked_train_mse_raw": best_payload["_picked_train_mse_raw"],
            "picked_val_mse_raw": best_payload["_picked_val_mse_raw"],
            "picked_overall_mse_raw": best_payload["_picked_overall_mse_raw"],
            "trials": all_rows,
        },
    }

### Scale, Train, Search, and Invert 

In [38]:
results = search_both_targets_LM(
    solver.historical_data,
    train_ratio=0.85,
    hidden_min=1,
    hidden_max=10,
    n_trials=30,
    base_seed=0,
    mu=1e-3,
    max_iters=50,
    device="cpu",
    # col_map={"i": "ffr"},  # uncomment if needed
)

best_y = results["y"]
best_pi = results["pi"]

print(f"Best y hidden units: {best_y['h']}  (avg val MSE raw over 30 trials = {best_y['metric']:.6g})")
print(f"Best pi hidden units: {best_pi['h']} (avg val MSE raw over 30 trials = {best_pi['metric']:.6g})")

Best y hidden units: 1  (avg val MSE raw over 30 trials = 0.092194)
Best pi hidden units: 1 (avg val MSE raw over 30 trials = 0.158342)


In [39]:
final_y = retrain_fixed_h_pick_best_overall(
    solver.historical_data,
    target_col="y",
    h_fixed=best_y["h"],
    train_ratio=0.85,
    n_trials=30,
    base_seed=0,
    mu=1e-3,
    max_iters=50,
    device="cpu",
    # col_map={"i": "ffr"},
)

final_pi = retrain_fixed_h_pick_best_overall(
    solver.historical_data,
    target_col="pi",
    h_fixed=best_pi["h"],
    train_ratio=0.85,
    n_trials=30,
    base_seed=999_999,  # separate seed stream from y
    mu=1e-3,
    max_iters=50,
    device="cpu",
    # col_map={"i": "ffr"},
)

print("Fixed h* retrain selection (y):",
      "h=", final_y["h"],
      "seed=", final_y["meta"]["picked_seed"],
      "overall MSE(raw)=", final_y["meta"]["picked_overall_mse_raw"])

print("Fixed h* retrain selection (pi):",
      "h=", final_pi["h"],
      "seed=", final_pi["meta"]["picked_seed"],
      "overall MSE(raw)=", final_pi["meta"]["picked_overall_mse_raw"])

Fixed h* retrain selection (y): h= 1 seed= 10017 overall MSE(raw)= 0.16120894666288096
Fixed h* retrain selection (pi): h= 1 seed= 1010009 overall MSE(raw)= 0.043220183047528525


### Figure 4 Output Gap Fit: Squared Errors (Replication)

### Figure 5 Inflation Fit: Squared Errors (Replication)

### Table 3 Economy Fit: Mean Squared Errors

In [41]:
from IPython.display import Markdown, display

svar_y_msq = solver.squared_errors_data["se_y_svar"].mean()
svar_pi_msq = solver.squared_errors_data["se_pi_svar"].mean()
ann_y_msq = final_y["meta"]["picked_overall_mse_raw"]
ann_pi_msq = final_pi["meta"]["picked_overall_mse_raw"]

table_three_str = fr"""
| Representation | MSE Output Gap   | MSE Inflation     | MSE Total                        |
|----------------|------------------|-------------------|----------------------------------|
| SVAR           | {svar_y_msq:.3f} | {svar_pi_msq:.3f} | {(svar_pi_msq+svar_y_msq)/2:.3f} |
| ANN            | {ann_y_msq:.3f}  | {ann_pi_msq:.3f}  | {(ann_y_msq+ann_pi_msq)/2:.3f}   |
"""

display(Markdown(table_three_str))


| Representation | MSE Output Gap   | MSE Inflation     | MSE Total                        |
|----------------|------------------|-------------------|----------------------------------|
| SVAR           | 0.211 | 0.032 | 0.122 |
| ANN            | 0.161  | 0.043  | 0.102   |


### PDP Functions

#### Figure 6: Partial Dependence Surface Plot - ANN Economy, Inflation

#### Figure 7: Partial Dependence Surface Plot - ANN Economy, Output Gap